# 05 — Classificar a RAIS por CEP (diagnóstico por CEP)

**Objetivo:** dizer, para cada CEP que aparece na RAIS (2017–2025), se ele já está espacializado (tem segmento na base oficial) ou precisa de verificação, e quanto da RAIS (estabelecimentos e vínculos) cada caso representa.

**Por que por CEP?** A RAIS tem ~330 mil linhas, mas poucos milhares de CEPs. Um CEP pode ter vários segmentos de rua; dividir a RAIS entre eles contaria o mesmo estabelecimento várias vezes. Por isso a espacialização é **por CEP** (linhas dissolvidas + um ponto no meio da linha).

**Classes (cada CEP da RAIS cai em exatamente uma):**

| Classe | Condição | Leitura |
|---|---|---|
| **A – via única** | CEP pareado a **1 logradouro** com extensão total até o limite (padrão 1 km) | espacializável com boa precisão |
| **B – via longa ou várias vias** | CEP pareado a mais de 1 logradouro **ou** extensão acima do limite | espacializável de forma aproximada (use para análise agregada) |
| **C – via ambígua** | CEP só aparece como candidato (`ceps_candidatos`) em vias da base sem `cep` definido | **verificar** |
| **D – órfão** | CEP existe nos Correios, mas não consta na coluna `cep` de nenhum segmento da base | **verificar** |
| **G – localizado pelo endereço** | CEP das classes C, D ou E com coordenada no CSV da etapa 04 (grandes usuários e CEPs sem trecho nem pontos do CNEFE, localizados por CNEFE, Geosanja, Nominatim ou API) | espacializável (`confianca` vem do CSV) |
| **P – pontos do CNEFE** | CEP das classes C, D ou E (`ausente_nos_correios` ou `cep_geral_sufixo_000`, como o 12249000 de São Francisco Xavier) **sem trecho na base**, mas com endereços no CNEFE: o ponto é o centro desses endereços, puxado para o segmento mais próximo | espacializável (`confianca` diz quão concentrados estão os pontos) |
| **E – fora dos Correios** | CEP não consta na lista de logradouros dos Correios (inválido, prefixo fora de 12, sufixo especial 9xx, sufixo 000...) | **verificar ou descartar** (coluna `motivo` explica; os grandes usuários localizados na etapa 04 viram G) |

Ordem: A/B → P → G. **C, D e E que sobram são os CEPs a espacializar à mão** (lista na etapa 06).

**Processo e confiança por CEP:**
- `processo`: de onde veio o ponto. A/B: `cep_origem` dos segmentos do CEP, resumido (`correios_geosanja`, `cnefe_segmento`, `manual`; vários separados por `|`); G: `localizacao_endereco` (a fonte está em `motivo` e `ponto_metodo`); P: `cnefe_pontos_cep`;
- `confianca`: A/B = a **pior** confiança entre os segmentos do CEP (`manual` conta como a melhor); G = coluna `confianca` do CSV da etapa 04; P = pela quantidade e dispersão dos pontos (`alta`: ≥ `min_pontos` locais e dispersão ≤ `mediana_suspeita_m`; `media`: ≥ 2 locais e dispersão ≤ `dispersao_media_m`; `baixa`: o resto).

**Entradas:**
- RAIS consolidada (etapa 01);
- **base de logradouros** `02_dados_editados\logradouros_cep.gpkg` (coluna `cep`; vários CEPs separados por `|`) — **obrigatória** (criada pela etapa 03);
- universo de CEPs dos Correios: `ponte_cep_logradouro.parquet` e `residuo_ceps_orfaos.csv` (etapa 02);
- `02_dados_editados\ceps_localizados.csv` (etapa 04): CEPs com coordenada viram **classe G**. Sem o CSV, nenhum CEP vira G;
- `cnefe_locais.parquet` (etapa 03): endereços do CNEFE, para a **classe P**. Sem ele, nenhum CEP vira P.

**Ciclo de correção:** corrija na base os CEPs que sobraram (lista da etapa 06) → rode de novo a partir desta etapa (`python executar_tudo.py --ciclo`). Nada aqui altera a base.

**Saídas** (pasta `03_processamento\05_classificar_rais_cep`):
- `diagnostico_cep.parquet` / `.csv` — uma linha por CEP da RAIS, com classe, motivo, extensão e totais
- `rais_cep_ano.parquet` — tabela longa (CEP × ano) com os totais da RAIS e a classe
- `rais_estab_sjc_classificada.parquet` — a RAIS (todas as linhas) com a classe do CEP
- `ceps_espacializados.gpkg` — camadas `ceps_linhas` (A e B) e `ceps_pontos` (A, B, G e P)
- `ranking_verificacao_ceps.csv` — CEPs das classes C, D e E, do maior para o menor impacto
- `residuo_rais_nao_espacializada.parquet` — linhas da RAIS sem ponto (C, D, E)
- `resumo_classes_ano.csv` — estabelecimentos e vínculos por classe e por ano

**Estrutura:** 1 Preparação · 2 Carregar entradas · 3 Informações por CEP · 4 Geometria por CEP · 5 Classificação · 6 Totais da RAIS · 7 Ranking e resíduos · 8 Exportação · 9 Resumo

## 1. Preparação e configuração
Caminhos e parâmetros vêm do `config.yaml`. Para mudar uma regra (limite de extensão, ano de referência), edite o `config.yaml`.

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "05_classificar_rais_cep"           # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import pyogrio                        # leitura de arquivos geográficos (GeoPackage)
import geopandas as gpd               # tabelas com geometria
import shapely                        # operações em linhas e pontos
import shapely.ops                    # ponto mais próximo de uma linha
from src import fontes                # arquivos editados (prioridade editado -> bruto)

# --- Entradas (config.yaml) -------------------------------------------------------------------------------
ARQ_RAIS = cfg.caminho("processamento.rais_consolidada")     # rais consolidada (etapa 01)
ARQ_PONTE = cfg.caminho("processamento.ponte")               # só para o universo de CEPs dos correios (etapa 02)
ARQ_ORFAOS = cfg.caminho("processamento.orfaos")             # só para o universo de CEPs dos correios (etapa 02)
ARQ_BASE = fontes.caminho_editado(cfg, "logradouros_cep")    # BASE OFICIAL editada (para com erro se não existir)
CAMADA_BASE = cfg.camada("editados.logradouros_cep")
ARQ_GRANDES = cfg.caminho("editados.ceps_localizados")       # CSV da etapa 04 (classe G); opcional
CRS_GRANDES = cfg.projeto("crs_graus")                       # lat/lon do CSV em graus (SIRGAS 2000)
ARQ_LOCAIS = cfg.caminho("processamento.cnefe_locais")       # locais do cnefe (etapa 03, classe P); opcional

# --- Parâmetros da classificação ------------------------------------------------------------------------------
LIMITE_EXTENSAO_M = cfg.parametro("classificacao.limite_extensao_m")       # extensão acima disso -> classe B
LIMITE_LOGRADOUROS = cfg.parametro("classificacao.limite_logradouros")     # mais logradouros que isso -> classe B
SUFIXO_ESPECIAL_MIN = cfg.parametro("localizacao.sufixo_especial_min")   # sufixo >= isso = grande usuário
ANO_REFERENCIA = cfg.parametro("classificacao.ano_referencia")             # None = ano mais recente da rais
CRS_ESPERADO = cfg.projeto("crs_metrico")                                  # CRS do geosanja (metros)
DIST_BUSCA_M = cfg.parametro("cnefe.dist_busca_m")                         # classe P: puxa o ponto para um segmento a até isso
MIN_PONTOS = cfg.parametro("cnefe.min_pontos")                             # classe P: mínimo de locais para confiança alta
DISPERSAO_ALTA_M = cfg.parametro("cnefe.mediana_suspeita_m")               # classe P: dispersão máxima para confiança alta
DISPERSAO_MEDIA_M = cfg.parametro("cnefe.dispersao_media_m")               # classe P: dispersão máxima para confiança média
CLASSES_VERIFICAR = ["C", "D", "E"]                   # classes que precisam de verificação (sem ponto)
CLASSES_ESPACIALIZADAS = ["A", "B", "G", "P"]         # classes que ganham ponto (G = localizado pelo endereço; P = pontos do cnefe)
ORDEM_CONFIANCA = {"baixa": 0, "sem_info": 0, "media": 1, "alta": 2, "manual": 3}   # para achar a "pior" confiança do CEP

log.info("Configuração carregada!")
for _nome, _arq in [("rais", ARQ_RAIS), ("Base oficial", ARQ_BASE), ("Ponte (correios)", ARQ_PONTE),
                    ("Órfãos (correios)", ARQ_ORFAOS), ("CEPs localizados (etapa 04)", ARQ_GRANDES), ("Locais do cnefe", ARQ_LOCAIS)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())
log.info("Pasta de saída: %s", PASTA_SAIDA)

In [ ]:
from src.validacao import validar_tabela      # bloco padrão de validação
from src.geo import para_gpkg                 # tipos compatíveis com GeoPackage


def ponto_central(geom):
    """Ponto do CEP: o MEIO da linha (50% do comprimento). Devolve (ponto, método, nº de partes).

    1. junta os segmentos que se tocam (line_merge) numa só linha;
    2. se sobrou UMA linha: ponto a 50% do comprimento dela;
    3. se sobraram várias partes soltas (CEP com ruas separadas): centroide do conjunto, "puxado" para a linha mais próxima.
    """
    try:
        partes = shapely.get_parts(shapely.line_merge(geom))            # lista de linhas já emendadas
        if len(partes) == 1:
            return shapely.line_interpolate_point(partes[0], 0.5, normalized=True), "meio_da_linha", 1
        return shapely.ops.nearest_points(geom, geom.centroid)[0], "centroide_ajustado", len(partes)
    except Exception:                                                    # geometria inesperada: usa o método antigo
        return geom.representative_point(), "ponto_representativo_reserva", 0


log.info("Funções auxiliares definidas!")

## 2. Carregar as entradas
Lê a RAIS e os resultados da Etapa 02, e confere o básico: colunas obrigatórias, CEP com 8 dígitos e CRS do Geosanja.

In [ ]:
log.info("Carregando a rais...")
rais = pd.read_parquet(ARQ_RAIS)
n_rais = len(rais)
colunas_rais = ["ano", "cep", "cep_valido", "cep_prefixo_fora_12", "ind_rais_negativa",
                "qtd_vinc_clt", "qtd_vinc_ativos", "qtd_vinc_estatutarios"]
faltam = [c for c in colunas_rais if c not in rais.columns]
assert not faltam, f"Colunas ausentes na rais: {faltam}"

# --- Padronização: CEP como texto sem espaços; "cep_chave" nunca é nulo (vazio = CEP ausente) ---
rais["cep"] = rais["cep"].astype("string").str.strip()
rais["cep_chave"] = rais["cep"].fillna("").astype(object)      # texto comum; vazio = CEP ausente
# --- Números: nulos viram 0 e tudo vira inteiro comum (evita problemas nas somas) ---
for col in ["qtd_vinc_clt", "qtd_vinc_ativos", "qtd_vinc_estatutarios"]:
    rais[col] = rais[col].fillna(0).astype("int64")
rais["ind_rais_negativa"] = rais["ind_rais_negativa"].fillna(0).astype("int64")
rais["ano"] = rais["ano"].astype("int64")
rais["cep_valido"] = rais["cep_valido"].fillna(False).astype(bool)
rais["cep_prefixo_fora_12"] = rais["cep_prefixo_fora_12"].fillna(False).astype(bool)

ANO_REF = int(rais["ano"].max()) if ANO_REFERENCIA is None else int(ANO_REFERENCIA)
log.info("rais: %d linhas | anos: %s | ano de referência: %d", n_rais, sorted(rais["ano"].unique().tolist()), ANO_REF)
log.info("CEPs distintos na rais: %d | CEP vazio: %d linhas", rais["cep_chave"].nunique(), int((rais["cep_chave"] == "").sum()))
fora_8 = int((~rais["cep_chave"].str.fullmatch(r"\d{8}")).sum())
(log.info if fora_8 == 0 else log.warning)("Linhas com CEP fora do formato de 8 dígitos: %d", fora_8)
validar_tabela("rais", rais, obrigatorias=["ano", "cep_chave"], linhas_entrada=n_rais)

log.info("Carregando a base oficial de logradouros com CEP...")
base = pyogrio.read_dataframe(ARQ_BASE, layer=CAMADA_BASE)                  # todas as colunas (a base pode ter colunas novas)
log.info("Segmentos na base: %d | CRS: %s", len(base), base.crs)
assert base.crs is not None and base.crs.to_string() == CRS_ESPERADO, f"CRS inesperado: {base.crs}"
validar_tabela("Base oficial (geometria)", base, chave=["id"], obrigatorias=["id"])
for col in ["tipo", "nome", "loteamento", "cep", "ceps_candidatos", "cep_origem", "cep_confianca"]:
    if col not in base.columns:                                              # base antiga sem a coluna: cria vazia
        log.warning("Coluna '%s' não existe na base: tratada como vazia.", col)
        base[col] = ""
    base[col] = base[col].astype(object).where(base[col].notna(), "").astype(str).str.strip()
base = base[["id", "tipo", "nome", "loteamento", "cep", "ceps_candidatos", "cep_origem", "cep_confianca", "geometry"]]

# --- Processo e confiança de cada segmento ----------------------------------------------------------------------
origem = base["cep_origem"].str.replace("automatico:", "", regex=False)       # versões antigas usavam o prefixo 'automatico:'
base["processo_seg"] = np.select(
    [origem.str.startswith("correios_geosanja") | origem.str.startswith("exato") | origem.isin(["iniciais", "fuzzy"]),
     origem.str.startswith("cnefe"), origem.str.lower() == "manual"],
    ["correios_geosanja", "cnefe_segmento", "manual"], default="sem_origem")
base["confianca_seg"] = np.where(base["processo_seg"] == "manual", "manual",
                                 np.where(base["cep_confianca"] != "", base["cep_confianca"], "sem_info"))
log.info("Segmentos com CEP por processo:\n%s", base.loc[base["cep"] != "", "processo_seg"].value_counts().to_string())

# --- Um par (segmento, CEP) por linha: separa a coluna 'cep' em '|' e limpa cada valor -------------------------
pares = base.loc[base["cep"] != "", ["id", "tipo", "nome", "cep_origem", "processo_seg", "confianca_seg", "cep"]].copy()
pares["cep"] = pares["cep"].str.split("|")
pares = pares.explode("cep")
pares["cep"] = pares["cep"].str.replace(r"\D", "", regex=True)               # tira hífen, ponto, espaço
fora_formato = int((pares["cep"].str.len() != 8).sum())
(log.info if fora_formato == 0 else log.warning)("Valores de 'cep' fora do formato de 8 dígitos na base (ignorados): %d", fora_formato)
if fora_formato:
    log.warning("Exemplos: %s", pares.loc[pares["cep"].str.len() != 8, "cep"].unique()[:10].tolist())
pares = pares[pares["cep"].str.len() == 8].drop_duplicates(["id", "cep"]).reset_index(drop=True)
pares["n_ceps_seg"] = pares.groupby("id")["cep"].transform("size")                 # quantos CEPs o segmento tem
pares["via"] = pares["tipo"] + " " + pares["nome"]
pares["metodo"] = pares["cep_origem"].str.replace("automatico:", "", regex=False).replace("", "sem_origem")
log.info("Pares (segmento, CEP): %d | segmentos com CEP: %d | CEPs distintos na base: %d",
         len(pares), pares["id"].nunique(), pares["cep"].nunique())
validar_tabela("Pares segmento-CEP", pares, chave=["id", "cep"], obrigatorias=["id", "cep"])

# --- Universo de CEPs dos correios (Etapa 02): pareados na ponte + órfãos ------------------------------------------------
ponte = pd.read_parquet(ARQ_PONTE, columns=["cep", "logradouro_correios", "bairro_correios"])
ponte["cep"] = ponte["cep"].astype(str)
orfaos_e1 = pd.read_csv(ARQ_ORFAOS, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
validar_tabela("Órfãos da Etapa 02 (universo dos correios)", orfaos_e1, obrigatorias=["cep"])
log.info("Entradas carregadas!")

## 3. Informações por CEP (base oficial + Correios)
Monta, para cada CEP, o que sabemos dele:
- **Correios:** logradouro e bairro (universo da Etapa 02);
- **Base oficial:** quantos segmentos e logradouros distintos o CEP tem, se divide a via com outros CEPs, e se o CEP **não consta nos Correios** (`cep_fora_correios`, útil para achar erro de digitação);
- **Órfãos:** CEPs dos Correios que nenhum segmento da base usa;
- **Ambíguos:** vias sem `cep` com `ceps_candidatos`: em quantas aparece e exemplos.

In [ ]:
log.info("Montando informações por CEP...")

# --- Universo dos correios (logradouro e bairro por CEP) ------------------------------------------------------------
info_ponte = (ponte.groupby("cep", as_index=False)
              .agg(logradouro_correios=("logradouro_correios", "first"), bairro_correios=("bairro_correios", "first")))
info_orf = (orfaos_e1.rename(columns={"logradouro_orig": "logradouro_correios", "bairro_orig": "bairro_correios"})
            [["cep", "logradouro_correios", "bairro_correios"]])
info_correios = pd.concat([info_ponte, info_orf]).drop_duplicates("cep")      # universo de CEPs dos correios
conjunto_correios = set(info_correios["cep"])

# --- Estatísticas dos CEPs que estão na base (um logradouro = tipo + nome) ----------------------------------------
pares["conf_ordem"] = pares["confianca_seg"].map(ORDEM_CONFIANCA).fillna(0)     # confiança em número (0 = pior)
stats_ponte = (pares.groupby("cep", as_index=False)
               .agg(n_segmentos=("id", "nunique"),
                    n_logradouros=("via", "nunique"),
                    compartilha_via=("n_ceps_seg", lambda s: bool((s > 1).any())),   # algum segmento do CEP tem outro CEP também
                    metodos=("metodo", lambda s: "|".join(sorted(set(s)))),
                    processo=("processo_seg", lambda s: "|".join(sorted(set(s)))),   # processos que deram os segmentos do CEP
                    conf_ordem=("conf_ordem", "min")))                              # a pior confiança entre os segmentos
inverso = {0: "baixa", 1: "media", 2: "alta", 3: "manual"}
stats_ponte["confianca"] = stats_ponte["conf_ordem"].map(inverso)
stats_ponte = stats_ponte.drop(columns="conf_ordem")
stats_ponte["cep_fora_correios"] = ~stats_ponte["cep"].isin(conjunto_correios)
n_fora = int(stats_ponte["cep_fora_correios"].sum())
(log.info if n_fora == 0 else log.warning)("CEPs na base que NÃO constam nos correios (possível erro de digitação): %d", n_fora)

# --- Órfãos = CEPs dos correios que não estão na coluna 'cep' de nenhum segmento ------------------------------------
conjunto_base = set(pares["cep"])
conjunto_orfaos = conjunto_correios - conjunto_base

# --- Candidatos em vias ambíguas: segmentos SEM cep definido e COM ceps_candidatos (um CEP por linha) ---------------------
amb_long = base.loc[(base["cep"] == "") & (base["ceps_candidatos"] != ""), ["id", "tipo", "nome", "loteamento", "ceps_candidatos"]].copy()
amb_long["cep"] = amb_long["ceps_candidatos"].str.split("|")
amb_long = amb_long.explode("cep")
amb_long["cep"] = amb_long["cep"].str.replace(r"\D", "", regex=True)
amb_long = amb_long[amb_long["cep"].str.len() == 8].copy()
amb_long["via_txt"] = (amb_long["tipo"] + " " + amb_long["nome"]
                       + np.where(amb_long["loteamento"] != "", " (" + amb_long["loteamento"] + ")", ""))
stats_amb = (amb_long.groupby("cep", as_index=False)
             .agg(n_vias_ambiguas=("id", "nunique"),
                  vias_ambiguas_exemplos=("via_txt", lambda s: " ; ".join(list(dict.fromkeys(s))[:5]))))   # até 5 exemplos
conjunto_ambiguos = set(stats_amb["cep"])
log.info("CEPs nos correios: %d | na base: %d | órfãos (correios sem segmento): %d | candidatos em vias ambíguas: %d",
         len(conjunto_correios), len(conjunto_base), len(conjunto_orfaos), len(conjunto_ambiguos))
validar_tabela("Informações dos correios por CEP", info_correios, chave=["cep"], obrigatorias=["cep"])
log.info("Informações por CEP montadas!")

## 4. Geometria por CEP
Só para os CEPs que **aparecem na RAIS** e foram pareados:
1. junta os segmentos de cada CEP (coluna `cep` da base oficial);
2. **dissolve** os segmentos em uma geometria por CEP;
3. mede a **extensão total** (metros, o CRS é métrico);
4. calcula o **ponto do CEP** (feito na exportação): o **meio da linha** (50% do comprimento). Se o CEP tiver ruas soltas, usa o centroide puxado para a linha mais próxima (`ponto_metodo` indica qual).

In [ ]:
log.info("Calculando a geometria por CEP...")
ceps_rais = set(rais["cep_chave"]) - {""}
pares_rais = pares[pares["cep"].isin(ceps_rais)][["id", "cep"]]
log.info("CEPs da rais presentes na base: %d (de %d distintos na rais)", pares_rais["cep"].nunique(), len(ceps_rais))

# --- Une cada par (segmento, CEP) à sua geometria (esquerda = camada geográfica, para manter a geometria) ---
seg = base[["id", "geometry"]].merge(pares_rais, on="id", how="inner")
# --- Uma geometria por CEP: junta todos os segmentos do CEP --------------------------------------------------
geom_cep = seg[["cep", "geometry"]].dissolve(by="cep", as_index=False)
geom_cep["extensao_m"] = geom_cep.geometry.length.round(1)
sem_geom = geom_cep.geometry.isna().sum() + geom_cep.geometry.is_empty.sum()
(log.info if sem_geom == 0 else log.warning)("CEPs com geometria vazia: %d", int(sem_geom))
log.info("CEPs com geometria: %d | extensão mediana: %.0f m | máxima: %.0f m",
         len(geom_cep), geom_cep["extensao_m"].median(), geom_cep["extensao_m"].max())
validar_tabela("Geometria por CEP", geom_cep, chave=["cep"], obrigatorias=["cep"])
log.info("Geometria por CEP calculada!")

## 5. Classificação dos CEPs da RAIS
Cada CEP da RAIS recebe **uma classe** (A–E) e um **motivo** que explica a decisão. A ordem de decisão:
1. tem segmento na base → **A** (1 logradouro e extensão ≤ limite) ou **B** (caso contrário);
2. senão, aparece como candidato em via ambígua → **C**;
3. senão, existe nos Correios → **D** (órfão);
4. senão → **E** (fora dos Correios), com o motivo: CEP vazio/inválido, prefixo fora de 12, sufixo especial, sufixo 000 ou "ausente nos Correios";
5. C, D ou E (`ausente_nos_correios` ou `cep_geral_sufixo_000`) com endereços no CNEFE → **P** (o motivo guarda a classe anterior);
6. C, D ou E que sobrou e tem coordenada no CSV da etapa 04 → **G** (localizado pelo endereço).

In [ ]:
log.info("Classificando os CEPs da rais...")

# --- Base: um CEP por linha, com os sinalizadores que vieram da rais -------------------------------------------
diag = (rais.groupby("cep_chave", as_index=False)
        .agg(cep_valido=("cep_valido", "first"), cep_prefixo_fora_12=("cep_prefixo_fora_12", "first"))
        .rename(columns={"cep_chave": "cep"}))
diag = (diag.merge(stats_ponte, on="cep", how="left")
            .merge(geom_cep[["cep", "extensao_m"]], on="cep", how="left")
            .merge(stats_amb, on="cep", how="left")
            .merge(info_correios, on="cep", how="left"))
diag["compartilha_via"] = diag["compartilha_via"].fillna(False).astype(bool)
diag["cep_fora_correios"] = diag["cep_fora_correios"].fillna(False).astype(bool)
diag["n_vias_ambiguas"] = diag["n_vias_ambiguas"].fillna(0).astype(int)
diag["tambem_candidato_ambiguo"] = (diag["n_vias_ambiguas"] > 0) & diag["n_segmentos"].notna()   # pareado, mas também aparece em via ambígua

# --- Condições (cada uma é uma coluna verdadeiro/falso) --------------------------------------------------------
pareado = diag["n_segmentos"].notna()
longa = diag["extensao_m"].fillna(np.inf) > LIMITE_EXTENSAO_M               # sem geometria conta como "longa" (cai em B)
varias = diag["n_logradouros"].fillna(0) > LIMITE_LOGRADOUROS
classe_a = pareado & ~longa & ~varias
classe_b = pareado & ~classe_a
classe_c = ~pareado & diag["cep"].isin(conjunto_ambiguos)
classe_d = ~pareado & ~classe_c & diag["cep"].isin(conjunto_orfaos)
diag["classe"] = np.select([classe_a, classe_b, classe_c, classe_d], ["A", "B", "C", "D"], default="E")

# --- Motivo (texto curto que explica a classe) ------------------------------------------------------------------
sufixo = diag["cep"].str[-3:]
motivo_b = np.where(longa & varias, "extensao_acima_do_limite+varios_logradouros",
            np.where(varias, "varios_logradouros", "extensao_acima_do_limite"))
motivo_e = np.select(
    [diag["cep"] == "", ~diag["cep_valido"], diag["cep_prefixo_fora_12"],
     sufixo.str.fullmatch(r"\d{3}") & (pd.to_numeric(sufixo, errors="coerce") >= SUFIXO_ESPECIAL_MIN),
     sufixo == "000"],
    ["cep_vazio", "cep_invalido", "prefixo_fora_de_12", "sufixo_especial_grande_usuario", "cep_geral_sufixo_000"],
    default="ausente_nos_correios")
diag["motivo"] = np.select([classe_a, classe_b, classe_c, classe_d],
                           ["via_unica_curta", motivo_b,
                            "via_ambigua_no_geosanja", "cep_sem_via_no_geosanja"],
                           default=motivo_e)

# --- Classe P: CEPs sem trecho na base, mas com endereços no cnefe ----------------------------------------------------------
pontos_p = pd.DataFrame(columns=["cep", "x_ponto", "y_ponto"])          # ponto de cada CEP da classe P (preenchido abaixo)
diag["n_locais_cnefe"] = np.nan
diag["dispersao_cnefe_m"] = np.nan
if ARQ_LOCAIS.exists():
    log.info("Classe P: buscando os CEPs sem trecho nos locais do cnefe...")
    loc = pd.read_parquet(ARQ_LOCAIS, columns=["cep", "x", "y"])
    # inclui CEP 000 fora dos correios: pode ser CEP geral de um distrito (ex.: 12249000, São Francisco Xavier); pontos espalhados -> confiança baixa
    elegivel = diag["classe"].isin(["C", "D"]) | ((diag["classe"] == "E") & diag["motivo"].isin(["ausente_nos_correios", "cep_geral_sufixo_000"]))
    loc = loc[loc["cep"].isin(set(diag.loc[elegivel, "cep"]))].copy()
    centro = loc.groupby("cep", as_index=False).agg(n_locais_cnefe=("x", "size"), x_med=("x", "median"), y_med=("y", "median"))
    loc = loc.merge(centro, on="cep")
    loc["d"] = np.hypot(loc["x"] - loc["x_med"], loc["y"] - loc["y_med"])         # distância de cada local ao centro (m)
    centro = centro.merge(loc.groupby("cep", as_index=False).agg(dispersao_cnefe_m=("d", "median")), on="cep")
    centro["dispersao_cnefe_m"] = centro["dispersao_cnefe_m"].round(1)
    centro["confianca"] = np.select(
        [(centro["n_locais_cnefe"] >= MIN_PONTOS) & (centro["dispersao_cnefe_m"] <= DISPERSAO_ALTA_M),
         (centro["n_locais_cnefe"] >= 2) & (centro["dispersao_cnefe_m"] <= DISPERSAO_MEDIA_M)],
        ["alta", "media"], default="baixa")
    mask_p = diag["cep"].isin(set(centro["cep"])) & elegivel
    info_p = centro.set_index("cep")
    diag.loc[mask_p, "motivo"] = "pontos_cnefe_do_cep (antes: " + diag.loc[mask_p, "classe"] + " " + diag.loc[mask_p, "motivo"] + ")"
    diag.loc[mask_p, "classe"] = "P"
    diag.loc[mask_p, "processo"] = "cnefe_pontos_cep"
    diag.loc[mask_p, "confianca"] = diag.loc[mask_p, "cep"].map(info_p["confianca"])
    diag.loc[mask_p, "n_locais_cnefe"] = diag.loc[mask_p, "cep"].map(info_p["n_locais_cnefe"])
    diag.loc[mask_p, "dispersao_cnefe_m"] = diag.loc[mask_p, "cep"].map(info_p["dispersao_cnefe_m"])
    pontos_p = centro.rename(columns={"x_med": "x_ponto", "y_med": "y_ponto"})[["cep", "x_ponto", "y_ponto"]]
    log.info("CEPs que viraram classe P: %d | confiança:\n%s", int(mask_p.sum()),
             diag.loc[mask_p, "confianca"].value_counts().to_string())
else:
    log.warning("Locais do cnefe não encontrados (%s): nenhum CEP vira P. Rode a etapa 03.", ARQ_LOCAIS.name)

# --- Classe G: CEPs localizados pelo endereço na etapa 04 (grandes usuários e CEPs sem trecho nem pontos do cnefe) ----------
grandes_ok = pd.DataFrame(columns=["cep", "lat", "lon", "fonte_coord", "confianca"])
if ARQ_GRANDES.exists():
    g = fontes.ler_csv_editado(ARQ_GRANDES)
    if "confianca" not in g.columns:
        g["confianca"] = ""
    g["cep"] = g["cep"].str.replace(r"\D", "", regex=True)
    for col in ["lat", "lon"]:
        g[col] = fontes.ler_coordenada(g[col])                                                  # aceita vírgula, ponto e o decimal estragado pelo Excel
    g = g[g["lat"].between(-90, 90) & g["lon"].between(-180, 180)]                              # só coordenadas válidas
    g.loc[g["fonte_coord"].str.lower() == "manual", "confianca"] = "manual"
    g.loc[g["confianca"] == "", "confianca"] = "baixa"                                         # sem informação: pior caso
    grandes_ok = g[["cep", "lat", "lon", "fonte_coord", "confianca"]].drop_duplicates("cep")
    log.info("CSV de CEPs localizados: %d CEPs com coordenada válida", len(grandes_ok))
else:
    log.warning("CSV de CEPs localizados não encontrado (%s): nenhum CEP vira classe G. Rode a etapa 04 e esta etapa de novo.", ARQ_GRANDES.name)
mask_g = diag["classe"].isin(["C", "D", "E"]) & diag["cep"].isin(set(grandes_ok["cep"]))   # A, B e P têm prioridade
info_g = grandes_ok.set_index("cep")
diag.loc[mask_g, "motivo"] = ("localizado_pelo_endereco:" + diag.loc[mask_g, "cep"].map(info_g["fonte_coord"])
                              + " (antes: " + diag.loc[mask_g, "classe"] + " " + diag.loc[mask_g, "motivo"] + ")")
diag.loc[mask_g, "classe"] = "G"
diag.loc[mask_g, "processo"] = "localizacao_endereco"
diag.loc[mask_g, "confianca"] = diag.loc[mask_g, "cep"].map(info_g["confianca"])
log.info("CEPs que viraram classe G (localizados pelo endereço): %d | confiança:\n%s", int(mask_g.sum()),
         diag.loc[mask_g, "confianca"].value_counts().to_string())
contagem = diag["classe"].value_counts().reindex(list("ABCDEGP"), fill_value=0)
log.info("CEPs da rais por classe:\n%s", contagem.to_string())
log.info("Motivos (classe E):\n%s", diag.loc[diag["classe"] == "E", "motivo"].value_counts().to_string())
validar_tabela("Classificação por CEP", diag, chave=["cep"], obrigatorias=["cep", "classe", "motivo"], linhas_entrada=rais["cep_chave"].nunique())
log.info("Classificação concluída!")

## 6. Totais da RAIS por CEP
- **CEP × ano:** estabelecimentos, estabelecimentos com RAIS negativa, estabelecimentos com vínculo ativo, e vínculos (CLT, ativos, estatutários);
- **Por CEP (série):** anos presentes e soma dos anos;
- **Ano de referência:** os mesmos totais só para o ano de referência (usados no ranking).

*Cada linha da RAIS é um estabelecimento (CNPJ/CEI) naquele ano.*

In [ ]:
log.info("Agregando a rais por CEP e ano...")
rais["_rais_negativa"] = rais["ind_rais_negativa"].eq(1).astype("int64")
rais["_com_vinculo"] = rais["qtd_vinc_ativos"].gt(0).astype("int64")

cep_ano = (rais.groupby(["cep_chave", "ano"], as_index=False)
           .agg(n_estab=("ano", "size"),
                n_estab_rais_negativa=("_rais_negativa", "sum"),
                n_estab_com_vinculo=("_com_vinculo", "sum"),
                vinc_clt=("qtd_vinc_clt", "sum"),
                vinc_ativos=("qtd_vinc_ativos", "sum"),
                vinc_estatutarios=("qtd_vinc_estatutarios", "sum"))
           .rename(columns={"cep_chave": "cep"}))
cep_ano = cep_ano.merge(diag[["cep", "classe", "motivo"]], on="cep", how="left")
validar_tabela("rais CEP × ano", cep_ano, chave=["cep", "ano"], obrigatorias=["cep", "ano", "classe"])

# --- Conferência: nada pode se perder na agregação -----------------------------------------------------------------
ok_estab = int(cep_ano["n_estab"].sum()) == n_rais
ok_vinc = int(cep_ano["vinc_ativos"].sum()) == int(rais["qtd_vinc_ativos"].sum())
(log.info if ok_estab else log.error)("[%s] estabelecimentos: %d na agregação = %d na rais", "OK" if ok_estab else "ERRO", int(cep_ano["n_estab"].sum()), n_rais)
(log.info if ok_vinc else log.error)("[%s] vínculos ativos: agregação = rais", "OK" if ok_vinc else "ERRO")

# --- Totais por CEP: série completa e ano de referência ----------------------------------------------------------
serie = (cep_ano.groupby("cep", as_index=False)
         .agg(n_anos=("ano", "nunique"), ano_min=("ano", "min"), ano_max=("ano", "max"),
              n_estab_soma_anos=("n_estab", "sum"), vinc_ativos_soma_anos=("vinc_ativos", "sum")))
ref = (cep_ano[cep_ano["ano"] == ANO_REF]
       [["cep", "n_estab", "n_estab_rais_negativa", "n_estab_com_vinculo", "vinc_ativos"]]
       .rename(columns={"n_estab": "n_estab_ref", "n_estab_rais_negativa": "n_estab_rais_negativa_ref",
                        "n_estab_com_vinculo": "n_estab_com_vinculo_ref", "vinc_ativos": "vinc_ativos_ref"}))
diag = diag.merge(serie, on="cep", how="left").merge(ref, on="cep", how="left")
for col in ["n_estab_ref", "n_estab_rais_negativa_ref", "n_estab_com_vinculo_ref", "vinc_ativos_ref"]:
    diag[col] = diag[col].fillna(0).astype("int64")                  # CEP que não existe no ano de referência = 0
diag["ano_ref"] = ANO_REF
log.info("Totais por CEP prontos: %d CEPs | ano de referência: %d", len(diag), ANO_REF)
log.info("Agregação concluída!")

## 7. Ranking de verificação e resíduos
- **Ranking:** CEPs das classes C, D e E ordenados por vínculos ativos no ano de referência, depois por vínculos na série e estabelecimentos. A coluna `pct_acum_vinc_ref` mostra quanto do total (dentro do ranking) já foi coberto — normalmente poucos CEPs concentram a maior parte.
- **RAIS classificada:** a RAIS original (todas as linhas) com a classe do CEP, para filtrar o que não tem vínculo espacial.
- **Resumo por classe e ano:** estabelecimentos e vínculos de cada classe.

In [ ]:
log.info("Montando ranking e resíduos...")

# --- Ranking de verificação ----------------------------------------------------------------------------------------
ranking = (diag[diag["classe"].isin(CLASSES_VERIFICAR)]
           .sort_values(["vinc_ativos_ref", "vinc_ativos_soma_anos", "n_estab_soma_anos"], ascending=False)
           .reset_index(drop=True))
ranking.insert(0, "posicao", np.arange(1, len(ranking) + 1))
total_vinc_ref = ranking["vinc_ativos_ref"].sum()
ranking["pct_acum_vinc_ref"] = (ranking["vinc_ativos_ref"].cumsum() / total_vinc_ref * 100).round(1) if total_vinc_ref else 0.0
colunas_ranking = ["posicao", "cep", "classe", "motivo", "logradouro_correios", "bairro_correios", "n_vias_ambiguas",
                   "vias_ambiguas_exemplos", "n_estab_ref", "n_estab_com_vinculo_ref", "vinc_ativos_ref",
                   "pct_acum_vinc_ref", "n_anos", "n_estab_soma_anos", "vinc_ativos_soma_anos", "ano_min", "ano_max"]
ranking = ranking[colunas_ranking]
validar_tabela("Ranking de verificação", ranking, chave=["cep"], obrigatorias=["cep", "classe"])

# --- rais classificada (todas as linhas) e resíduo por linha ---------------------------------------------------------
rais_classificada = rais.merge(
    diag[["cep", "classe", "motivo"]].rename(columns={"cep": "cep_chave", "classe": "classe_cep", "motivo": "motivo_cep"}),
    on="cep_chave", how="left", validate="many_to_one").drop(columns=["_rais_negativa", "_com_vinculo"])      # mantém cep_chave (CEP limpo): é a chave da Etapa 06
rais_classificada["espacializavel"] = rais_classificada["classe_cep"].isin(CLASSES_ESPACIALIZADAS)
validar_tabela("rais classificada", rais_classificada, obrigatorias=["classe_cep"], linhas_entrada=n_rais)
assert len(rais_classificada) == n_rais, "A junção mudou o número de linhas da rais!"
res_rais_nao_espacializada = rais_classificada[~rais_classificada["espacializavel"]].reset_index(drop=True)

# --- Resumo por classe e ano ------------------------------------------------------------------------------------------
resumo = (cep_ano.groupby(["ano", "classe"], as_index=False)
          .agg(n_ceps=("cep", "nunique"), n_estab=("n_estab", "sum"),
               n_estab_com_vinculo=("n_estab_com_vinculo", "sum"), vinc_ativos=("vinc_ativos", "sum")))
tot_ano = resumo.groupby("ano")[["n_estab", "vinc_ativos"]].transform("sum")
resumo["pct_estab"] = (resumo["n_estab"] / tot_ano["n_estab"] * 100).round(1)
resumo["pct_vinc_ativos"] = (resumo["vinc_ativos"] / tot_ano["vinc_ativos"].where(tot_ano["vinc_ativos"] > 0) * 100).round(1)

log.info("Ranking: %d CEPs a verificar | resíduo da rais: %d linhas (%.1f%%)",
         len(ranking), len(res_rais_nao_espacializada), len(res_rais_nao_espacializada) / n_rais * 100)
log.info("Resumo por classe no ano de referência (%d):\n%s", ANO_REF,
         resumo[resumo["ano"] == ANO_REF].drop(columns="ano").to_string(index=False))
log.info("Ranking e resíduos montados!")

## 8. Exportação
Salva tabelas em parquet (rápido), relatórios em CSV (abre direto no Excel, separador `;`) e as camadas de CEP em GeoPackage (linhas: A e B; pontos: A, B, G e P).

In [ ]:
log.info("Exportando resultados...")

# --- Tabelas ---------------------------------------------------------------------------------------------------------
colunas_diag = ["cep", "classe", "motivo", "processo", "confianca", "logradouro_correios", "bairro_correios", "n_segmentos",
                "n_logradouros", "extensao_m", "compartilha_via", "cep_fora_correios", "metodos", "n_locais_cnefe",
                "dispersao_cnefe_m", "n_vias_ambiguas", "tambem_candidato_ambiguo",
                "vias_ambiguas_exemplos", "cep_valido", "cep_prefixo_fora_12", "n_anos", "ano_min", "ano_max",
                "n_estab_soma_anos", "vinc_ativos_soma_anos", "ano_ref", "n_estab_ref", "n_estab_rais_negativa_ref",
                "n_estab_com_vinculo_ref", "vinc_ativos_ref"]
diag = diag[colunas_diag].sort_values(["classe", "vinc_ativos_ref"], ascending=[True, False]).reset_index(drop=True)
diag.to_parquet(PASTA_SAIDA / "diagnostico_cep.parquet", index=False)
diag.to_csv(PASTA_SAIDA / "diagnostico_cep.csv", sep=";", index=False, encoding="utf-8-sig")
cep_ano.to_parquet(PASTA_SAIDA / "rais_cep_ano.parquet", index=False)
rais_classificada.to_parquet(PASTA_SAIDA / "rais_estab_sjc_classificada.parquet", index=False)
res_rais_nao_espacializada.to_parquet(PASTA_SAIDA / "residuo_rais_nao_espacializada.parquet", index=False)
ranking.to_csv(PASTA_SAIDA / "ranking_verificacao_ceps.csv", sep=";", index=False, encoding="utf-8-sig")
resumo.to_csv(PASTA_SAIDA / "resumo_classes_ano.csv", sep=";", index=False, encoding="utf-8-sig")
log.info("Tabelas salvas!")

# --- Camadas geográficas (só A e B): linhas dissolvidas + ponto no meio da linha ----------------------------------------
log.info("Gravando o GeoPackage...")
atributos = diag[diag["classe"].isin(CLASSES_ESPACIALIZADAS)].drop(columns=["extensao_m"])
linhas = geom_cep[["cep", "extensao_m", "geometry"]].merge(atributos, on="cep", how="inner")      # inner: só A e B
resultado_pontos = [ponto_central(g) for g in linhas.geometry]                                       # (ponto, método, nº de partes) por CEP
linhas["n_partes"] = [r[2] for r in resultado_pontos]                                               # quantas linhas soltas o CEP tem
linhas["ponto_metodo"] = [r[1] for r in resultado_pontos]
pontos = linhas.copy()
pontos["geometry"] = gpd.GeoSeries([r[0] for r in resultado_pontos], index=linhas.index, crs=linhas.crs)   # camada de pontos
# --- Classe G: pontos dos CEPs localizados pelo endereço (vêm do CSV da etapa 04; não têm linha de rua) ------------------------
atrib_g = diag[diag["classe"] == "G"].merge(grandes_ok[["cep", "lat", "lon", "fonte_coord"]], on="cep", how="inner")
if len(atrib_g):
    pontos_g = atrib_g.drop(columns=["lat", "lon", "fonte_coord"]).copy()
    pontos_g["extensao_m"] = np.nan
    pontos_g["n_partes"] = 0
    pontos_g["ponto_metodo"] = "endereco:" + atrib_g["fonte_coord"].replace("", "sem_fonte").values
    pontos_g["geometry"] = gpd.GeoSeries(gpd.points_from_xy(atrib_g["lon"], atrib_g["lat"]), index=pontos_g.index,
                                         crs=CRS_GRANDES).to_crs(linhas.crs)                       # graus -> metros (EPSG:31983)
    pontos = gpd.GeoDataFrame(pd.concat([pontos, pontos_g.reindex(columns=pontos.columns)], ignore_index=True),
                              geometry="geometry", crs=linhas.crs)
    log.info("Pontos localizados pelo endereço (classe G) adicionados à camada ceps_pontos: %d", len(pontos_g))
# --- Classe P: centro dos endereços do cnefe, puxado para o segmento mais próximo da base (até dist_busca_m) ----------------
atrib_p = diag[diag["classe"] == "P"].merge(pontos_p, on="cep", how="inner")
if len(atrib_p):
    log.info("Calculando os pontos da classe P...")
    centros = gpd.GeoDataFrame(atrib_p[["cep"]], geometry=gpd.points_from_xy(atrib_p["x_ponto"], atrib_p["y_ponto"]),
                               crs=linhas.crs)                                                  # x/y já estão em metros
    perto = gpd.sjoin_nearest(centros, base[["id", "geometry"]], how="left", max_distance=DIST_BUSCA_M,
                              distance_col="dist_segmento").drop_duplicates("cep")          # segmento mais próximo (ou nenhum)
    perto = perto.set_index("cep").loc[atrib_p["cep"].to_numpy()]                            # mesma ordem de atrib_p
    geom_base = dict(zip(base["id"], base.geometry))                                          # id -> linha
    ponto_final, metodo_p = [], []
    for pt, id_seg in zip(perto.geometry, perto["id"]):                                       # laço curto (só CEPs da classe P)
        if pd.notna(id_seg):
            ponto_final.append(shapely.ops.nearest_points(geom_base[id_seg], pt)[0])         # ponto da linha mais perto do centro
            metodo_p.append("cnefe_pontos_cep:segmento_mais_proximo")
        else:
            ponto_final.append(pt)                                                            # nenhum segmento perto: fica o centro
            metodo_p.append("cnefe_pontos_cep:centro_dos_pontos")
    pontos_pp = atrib_p.drop(columns=["x_ponto", "y_ponto"]).copy()
    pontos_pp["extensao_m"] = np.nan
    pontos_pp["n_partes"] = 0
    pontos_pp["ponto_metodo"] = metodo_p
    pontos_pp["geometry"] = gpd.GeoSeries(ponto_final, index=pontos_pp.index, crs=linhas.crs)
    pontos = gpd.GeoDataFrame(pd.concat([pontos, pontos_pp.reindex(columns=pontos.columns)], ignore_index=True),
                              geometry="geometry", crs=linhas.crs)
    log.info("Pontos da classe P adicionados à camada ceps_pontos: %d (puxados para um segmento: %d)!", len(pontos_pp),
             sum(m.endswith("segmento_mais_proximo") for m in metodo_p))
log.info("Método dos pontos:\n%s", pontos["ponto_metodo"].value_counts().to_string())

arq_gpkg = PASTA_SAIDA / "ceps_espacializados.gpkg"
if arq_gpkg.exists():
    arq_gpkg.unlink()                                                                               # recomeça o arquivo do zero
para_gpkg(linhas).to_file(arq_gpkg, layer="ceps_linhas", driver="GPKG", engine="pyogrio")
para_gpkg(pontos).to_file(arq_gpkg, layer="ceps_pontos", driver="GPKG", engine="pyogrio", mode="a")
log.info("GeoPackage salvo: %s | camadas: ceps_linhas (%d, só A e B), ceps_pontos (%d, A, B, G e P)", arq_gpkg.name, len(linhas), len(pontos))
validar_tabela("Camada ceps_linhas", linhas, chave=["cep"], obrigatorias=["cep", "classe"])
log.info("Exportação concluída!")

## 9. Resumo
Quanto da RAIS já está espacializável, quanto precisa de verificação e onde está a concentração.

In [ ]:
log.info("Resumo final...")
def _pct(parte, total):
    """Percentual seguro (evita divisão por zero)."""
    return (parte / total * 100) if total else 0.0

ref_tab = cep_ano[cep_ano["ano"] == ANO_REF]
n_estab_ref = int(ref_tab["n_estab"].sum())
vinc_ref = int(ref_tab["vinc_ativos"].sum())
log.info("=== Ano de referência: %d | %d estabelecimentos | %d vínculos ativos ===", ANO_REF, n_estab_ref, vinc_ref)
for classe in list("ABGPCDE"):
    t = ref_tab[ref_tab["classe"] == classe]
    log.info("  Classe %s: %5d CEPs | %6d estab. (%.1f%%) | %7d vínculos (%.1f%%)", classe, t["cep"].nunique(),
             int(t["n_estab"].sum()), _pct(t["n_estab"].sum(), n_estab_ref),
             int(t["vinc_ativos"].sum()), _pct(t["vinc_ativos"].sum(), vinc_ref))
espac = ref_tab[ref_tab["classe"].isin(CLASSES_ESPACIALIZADAS)]
log.info("Espacializável (A+B+G+P): %.1f%% dos estabelecimentos e %.1f%% dos vínculos", _pct(espac["n_estab"].sum(), n_estab_ref), _pct(espac["vinc_ativos"].sum(), vinc_ref))
log.info("Só A (boa precisão): %.1f%% dos estabelecimentos e %.1f%% dos vínculos",
         _pct(ref_tab[ref_tab["classe"] == "A"]["n_estab"].sum(), n_estab_ref), _pct(ref_tab[ref_tab["classe"] == "A"]["vinc_ativos"].sum(), vinc_ref))

# --- Concentração: quantos CEPs do ranking cobrem 80% dos vínculos a verificar -------------------------------------------
com_vinc = ranking[ranking["vinc_ativos_ref"] > 0]
if len(com_vinc):
    n80 = int((com_vinc["pct_acum_vinc_ref"] < 80).sum()) + 1
    log.info("Ranking: %d CEPs cobrem 80%% dos vínculos a verificar (de %d CEPs com vínculos no ano de referência)", n80, len(com_vinc))
log.info("Top 15 do ranking:\n%s", ranking.head(15)[["posicao", "cep", "classe", "motivo", "logradouro_correios", "n_estab_ref", "vinc_ativos_ref"]].to_string(index=False))

log.info("Arquivos em %s:", PASTA_SAIDA)
for arq in sorted(PASTA_SAIDA.iterdir()):
    log.info("  %s (%.0f KB)", arq.name, arq.stat().st_size / 1024)
log.info("Etapa 05 concluída!")